# Lab 11: Variational Autoencoders and Latent Spaces

            **Duration:** 3 hours  
            **Lecture alignment:** Week 11 — Variational autoencoders  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Implement VAE reparameterization and ELBO components.
- Train a compact VAE and inspect reconstructions.
- Analyze latent organization and the reconstruction–regularization trade-off.

            ## Three-hour activity plan

            - 0–30 min: autoencoder baseline and probabilistic notation
- 30–70 min: encoder, reparameterization, decoder
- 70–125 min: ELBO implementation and training
- 125–160 min: reconstruction/latent analysis
- 160–180 min: checks and comparison


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20271
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_11")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_11"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 11, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

As KL weight increases, predict what happens to reconstruction sharpness and latent organization. Which plot would support each claim?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Build a small image distribution


In [ ]:
def make_tiny_images(n,size=8):
    labels=torch.randint(0,3,(n,));images=.03*torch.rand(n,1,size,size)
    for i,label in enumerate(labels.tolist()):
        if label==0:images[i,0,2:6,3:5]+=.9
        elif label==1:images[i,0,3:5,1:7]+=.9
        else:
            for j in range(1,7):images[i,0,j,j]+=.9
        images[i]=torch.roll(images[i],(int(torch.randint(-1,2,(1,))),int(torch.randint(-1,2,(1,)))),(1,2))
    return images.clamp(0,1),labels
images,labels=make_tiny_images(900 if FAST_MODE else 4000)
split=int(.8*len(images));train_images,test_images=images[:split],images[split:];train_labels,test_labels=labels[:split],labels[split:]


## Activity 2 — Reparameterization and ELBO training


In [ ]:
class VAE(nn.Module):
    def __init__(self,latent=2):
        super().__init__();self.enc=nn.Sequential(nn.Linear(64,40),nn.ReLU());self.mu=nn.Linear(40,latent);self.logvar=nn.Linear(40,latent)
        self.dec=nn.Sequential(nn.Linear(latent,40),nn.ReLU(),nn.Linear(40,64),nn.Sigmoid())
    def encode(self,x):
        h=self.enc(x.flatten(1));return self.mu(h),self.logvar(h)
    def reparameterize(self,mu,logvar):return mu+torch.randn_like(mu)*torch.exp(.5*logvar)
    def forward(self,x):
        mu,logvar=self.encode(x);z=self.reparameterize(mu,logvar);return self.dec(z).reshape(-1,1,8,8),mu,logvar
vae=VAE().to(DEVICE);opt=torch.optim.Adam(vae.parameters(),lr=.008);loader=DataLoader(TensorDataset(train_images),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));history=[]
for _ in range(16 if FAST_MODE else 50):
    total=0;vae.train()
    for (xb,) in loader:
        xb=xb.to(DEVICE);opt.zero_grad();recon,mu,logvar=vae(xb)
        recon_loss=F.binary_cross_entropy(recon,xb,reduction="sum")/len(xb);kl=-.5*torch.sum(1+logvar-mu.square()-logvar.exp())/len(xb)
        loss=recon_loss+.15*kl;loss.backward();opt.step();total+=loss.item()*len(xb)
    history.append(total/len(train_images))


## Activity 3 — Reconstructions, latent space, and interpolation


In [ ]:
vae.eval()
with torch.no_grad():recon,mu,logvar=vae(test_images.to(DEVICE));mu=mu.cpu();recon=recon.cpu()
fig,axes=plt.subplots(3,8,figsize=(10,4.2))
for j in range(8):axes[0,j].imshow(test_images[j,0],cmap="gray",vmin=0,vmax=1);axes[1,j].imshow(recon[j,0],cmap="gray",vmin=0,vmax=1)
for ax in axes[:2].flat:ax.axis("off")
axes[0,0].set_ylabel("input");axes[1,0].set_ylabel("recon")
axes[2,0].plot(history);axes[2,0].set_title("ELBO loss")
for j in range(1,8):axes[2,j].axis("off")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"vae_reconstructions.png",dpi=150);plt.show()
fig,ax=plt.subplots(figsize=(5,4));scatter=ax.scatter(mu[:,0],mu[:,1],c=test_labels,s=12,cmap="viridis");fig.colorbar(scatter,ax=ax);ax.set_title("VAE latent means");fig.tight_layout()
fig.savefig(ARTIFACT_DIR/"vae_latent.png",dpi=150);plt.show();torch.save(vae.state_dict(),ARTIFACT_DIR/"vae.pt")


## Automated checks


In [ ]:
assert recon.shape==test_images.shape and mu.shape[1]==2
assert torch.isfinite(mu).all() and history[-1]<history[0]
assert recon.min()>=0 and recon.max()<=1
assert (ARTIFACT_DIR/"vae.pt").exists()
print("All Lab 11 checks passed.")


## Deliverables

                - VAE implementation with reconstruction and KL terms
- Reconstruction and latent artifacts
- Saved VAE state and trade-off explanation

                Submit the executed notebook and the files created in `/content/artifacts/lab_11/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    z=torch.linspace(-2,2,12).unsqueeze(1).repeat(1,2).to(DEVICE)
    with torch.no_grad():samples=vae.dec(z).reshape(-1,1,8,8).cpu()
    print("Generated",len(samples),"latent traversal samples")
else:
    print("Extension disabled: latent traversal, conditional VAE, or beta-VAE comparison.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
